# This part will be used to webscrape data needed for NLP sentiments
# NLP sentiments will be processed here before it used on the main project

### Extraction Loop

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor 
import re
import nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
import time
import random

def scrape_page(page_num):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    if page_num == 1:
        url = 'https://www.bworldonline.com/banking-finance/'
    else:
        url = f'https://www.bworldonline.com/banking-finance/page/{page_num}/'
    
    # Random delay to avoid rate limiting
    time.sleep(random.uniform(0.5, 1.5))
    
    for attempt in range(3):  # Retry up to 3 times
        try:
            response = requests.get(url, headers=headers, timeout=20)  # Increased timeout
            if response.status_code == 429:  # Too Many Requests
                time.sleep(5 * (attempt + 1))  # Back off longer each retry
                continue
            if response.status_code != 200:
                return []
            
            soup = BeautifulSoup(response.text, 'html.parser')
            containers = soup.find_all('div', class_='td_module_10')
            page_data = []
            
            for box in containers:
                try:
                    headline_tag = box.find('h3').find('a')
                    headline = headline_tag.get_text(strip=True)
                    link = headline_tag.get('href')
                    parts = link.split('/')
                    if len(parts) >= 7:  # Guard against unexpected URL formats
                        date_val = f'{parts[4]}-{parts[5]}-{parts[6]}'
                    else:
                        date_val = 'unknown'
                    page_data.append({'date': date_val, 'headline': headline})
                except (AttributeError, IndexError):
                    continue  # Skip malformed articles silently
            
            return page_data
        
        except requests.exceptions.Timeout:
            print(f'Page {page_num} timed out (attempt {attempt + 1}/3)')
            time.sleep(3 * (attempt + 1))
        except Exception as e:
            print(f'Page {page_num} failed: {e}')
            return []
    
    print(f'Page {page_num} permanently failed after 3 attempts')
    return []

# Manager
pages = range(1, 801)
all_headlines = []
print('Start Scraping')
start_time = time.time()

with ThreadPoolExecutor(max_workers=3) as executor:  # Reduced from 10 → 3
    results = list(executor.map(scrape_page, pages))

for i in results:
    all_headlines.extend(i)

end_time = time.time()
print(f'Finished! Scraped {len(all_headlines)} headlines in {round(end_time - start_time, 2)} seconds.')

Start Scraping
Page 375 timed out (attempt 1/3)
Page 407 timed out (attempt 1/3)
Page 436 timed out (attempt 1/3)


In [ ]:
df = pd.DataFrame(all_headlines)
df_cleaned = df.drop_duplicates(subset = ['headline'], keep='first')
df_cleaned.reset_index(drop = True)
df = df_cleaned[df_cleaned['date'] >= '2021-01-01']
df['date'] = pd.to_datetime(df['date'])
df['headline_clean'] = df['headline'].str.lower()

In [ ]:
def clean_finance_text(text):
    text = text.lower()

    text = re.sub(r'(?<!\d)[.,:!](?!\d) | [^\s\w.]', '', text)
    return text

df['headline_clean'] = df['headline_clean'].apply(clean_finance_text)

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split() 
    filtered_words = [w for w in words if w not in stop_words]
    return ' '.join(filtered_words)

df['headline_clean'] = df['headline_clean'].apply(remove_stopwords)

df[['headline', 'headline_clean']]

finbert = pipeline('sentiment-analysis', model = 'ProsusAI/finbert')

In [ ]:
def get_finbertscore(headline):
    result = finbert(headline, top_k = None)
    scores = {res['label']: res['score'] for res in result}
    return scores['positive'] - scores['negative']

df['sentiment'] = df['headline_clean'].apply(get_finbertscore)